In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from unsloth import FastLanguageModel
import torch

# ============================
# Base model and configuration
# ============================
model_name    = "unsloth/Qwen3-8B-unsloth-bnb-4bit"
SEED          = 69
MAX_SEQ_LENGTH = 1024

# ============================
# Load model and tokenizer for LoRA finetuning
# ============================
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = model_name,
    max_seq_length  = MAX_SEQ_LENGTH,   # should match your dataset needs
    load_in_4bit    = True,             # required for 4bit LoRA finetuning
    load_in_8bit    = False,
    full_finetuning = False,            # keep this False for LoRA training
)

# ============================
# Prepare model for LoRA / DoRA finetuning
# ============================
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    lora_alpha = 64,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    use_rslora = False,    # keep False unless you explicitly want RS LoRA
    use_dora = True,       # enables DoRA
    loftq_config = None,   # no LoftQ quantization config
)

print("Model loaded and ready for LoRA finetuning.")


In [ ]:
import pandas as pd
from datasets import Dataset

# ---------- Paths ----------
OUTPUT_DIR = "/content/drive/MyDrive/qwen3-8b-unsloth-dora"

CSV_PATH   = "/content/drive/MyDrive/train.csv"


print(f"Loading dataset from: {CSV_PATH}")
# ---------- Load CSV ----------
df = pd.read_csv(CSV_PATH)

# ---------- Validate Columns ----------
required_cols = {"input_finding", "output_disease"}
missing = required_cols - set(df.columns)

if missing:
    raise ValueError(f"Missing required columns in CSV: {missing}")

# ---------- Clean Columns ----------
df = df[["input_finding", "output_disease"]].dropna()

df["input_finding"]  = df["input_finding"].astype(str).str.strip()
df["output_disease"] = df["output_disease"].astype(str).str.strip()

print("Number of rows after filtering:", len(df))
print("Columns:", list(df.columns))
display(df.head())

# ---------- Build Allowed Disease Label List ----------
disease_labels = (
    df["output_disease"]
      .astype(str)
      .str.split(",")
      .explode()
      .str.strip()
      .dropna()
)

# Remove empty entries
disease_labels = disease_labels[disease_labels != ""]

disease_labels = sorted(disease_labels.unique().tolist())
allowed_labels = ", ".join(disease_labels)

print("Number of unique disease labels:", len(disease_labels))

preview = allowed_labels[:300]
print("Allowed labels preview:", preview, "..." if len(allowed_labels) > 300 else "")

# ---------- Convert to Hugging Face Dataset ----------
hf_dataset = Dataset.from_pandas(df, preserve_index=False)
print(hf_dataset)


In [ ]:
import numpy as np

# ------------------------
# System prompt
# ------------------------
system_prompt = (
    "You are a clinical Named Entity Recognition (NER) and multi-label classification model. "
    "Read the abdominal radiology findings and identify all diseases that are present. "
    "Use only disease names from the allowed disease label list. "
    "Return the diseases as a comma separated list using the exact wording from the list. "
    "If none apply, output: No acute abnormality"
    f"Allowed disease label list: {allowed_labels}."
)

# ------------------------
# Formatting function
# ------------------------
def format_batch(batch):
    texts = []

    for finding, labels in zip(batch["input_finding"], batch["output_disease"]):
        finding = str(finding).strip()
        labels = str(labels).strip()

        messages = [
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": (
                    "Clinical findings:\n"
                    f"{finding}\n\n"
                    "List all diseases present using only labels from the allowed disease list. "
                    "Separate multiple diseases with commas. If none apply, output: No acute abnormality"
                ),
            },
            {"role": "assistant", "content": labels},
        ]

        # Convert chat messages to plain training text
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
        )
        texts.append(text)

    return {"text": texts}


print("Applying chat template to the dataset...")

processed_dataset = hf_dataset.map(
    format_batch,
    batched=True,
    remove_columns=hf_dataset.column_names,
)

print("Example processed text:")
example_text = processed_dataset[0]["text"]
print(example_text)

# -------------------------------
# Compute token length statistics
# -------------------------------
lengths = []

for sample in processed_dataset["text"]:
    encoded = tokenizer(
        sample,
        add_special_tokens=False,
    )
    lengths.append(len(encoded["input_ids"]))

lengths = np.array(lengths)

print("\nToken length statistics for training texts:")
print("Number of samples:", len(lengths))
print("Min length:", int(lengths.min()))
print("Max length:", int(lengths.max()))
print("Mean length:", float(lengths.mean()))
print("Median length (50th percentile):", int(np.percentile(lengths, 50)))
print("90th percentile:", int(np.percentile(lengths, 90)))
print("95th percentile:", int(np.percentile(lengths, 95)))
print("99th percentile:", int(np.percentile(lengths, 99)))

example_len = len(
    tokenizer(example_text, add_special_tokens=False)["input_ids"]
)
print("\nToken length of example 0:", example_len)


In [ ]:
from trl import SFTTrainer, SFTConfig

# Training hyperparameters
BATCH_SIZE     = 2           # Works on most 16–24 GB GPUs with Qwen3-8B 4bit LoRA
GRAD_ACCUM     = 4           # Effective batch size = 2 × 4 = 8 (good stability, low VRAM)
EPOCHS         = 4           # Dataset is small (1236 samples); 3 epochs is ideal
LR             = 2e-4        # Standard LoRA LR for 8B models; adjust lower if loss becomes unstable

print("Setting up training configuration...")

# Prefer bf16 when supported (more stable, faster).
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_args = SFTConfig(

    # Max tokens per sequence. Covers all samples (max ≈ 1039 tokens),
    # while being friendly to Qwen3-8B VRAM footprint.
    max_seq_length = MAX_SEQ_LENGTH,

    # Column name containing full training text after apply_chat_template.
    dataset_text_field = "text",

    # Real batch size per device/GPU. Qwen3-8B 4bit LoRA typically supports 1–2.
    per_device_train_batch_size = BATCH_SIZE,

    # Accumulate gradients to simulate larger effective batch without more memory.
    gradient_accumulation_steps = GRAD_ACCUM,

    # Number of full passes through the dataset. Works well with 1236 samples.
    num_train_epochs = EPOCHS,

    # Learning rate for LoRA adapter weights. 2e-4 is aggressive but stable for 4bit LoRA.
    learning_rate = LR,

    # Warmup steps prevent spikes in early training.
    warmup_steps = 50,

    # Logs progress every 10 steps.
    logging_steps = 10,

    # Saves a checkpoint at the end of each epoch.
    save_strategy = "epoch",

    # Output folder for checkpoints, logs, and final LoRA weights.
    output_dir = OUTPUT_DIR,

    # Use memory-efficient 8-bit AdamW optimizer (important for 8B model).
    optim = "adamw_8bit",

    # Whether to use bf16. Preferred when GPU supports it (A100, 4090, etc).
    bf16 = use_bf16,

    # Fallback to fp16 when bf16 is not supported.
    fp16 = not use_bf16,

    # Random seed for reproducible training.
    seed = SEED,
)

trainer = SFTTrainer(
    model         = model,              # Qwen3-8B 4bit LoRA-ready model
    tokenizer     = tokenizer,          # Must match model
    train_dataset = processed_dataset,  # Your processed dataset with template-applied text
    args          = training_args,
    packing       = True,               # Better GPU utilization when samples are shorter than max length
)

print("Trainer is ready.")


# There are 1236 samples, batch size 2, gradient accumulation 4, epochs 4.
# Effective batch size per optimizer step = 2 × 4 = 8 samples.
# Steps per epoch
# 1236 ÷ 8 = 154.5 → ceil to 155 steps per epoch.
# Total training steps
# 155 steps/epoch × 4 epochs = 620 optimizer steps in total.

# At each log, the loss shown is the average loss from all the small groups of 2 samples (BATCH_SIZE = 2)
# that were processed one after another and combined before performing a single optimizer update.


In [ ]:
import os

# Start training loop
print("Starting training...")
trainer.train()
print("Training completed.")

print("Saving fine tuned model and tokenizer to:", OUTPUT_DIR)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save LoRA adapter weights and config
model.save_pretrained(OUTPUT_DIR)

# Save tokenizer with the chat template and special tokens
tokenizer.save_pretrained(OUTPUT_DIR)

print("Done.")
